# Feature Engineering for Medical Insurance Cost Prediction

In [19]:
import numpy as np
import pandas as pd

## Load Cleaned Data

In [20]:
df = pd.read_csv('../data/insurance_clean.csv')

In [21]:
df_engineered = df.copy()
df_engineered.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [22]:
print(f"Rows: {df_engineered.shape[0]:,}")
print(f"Columns: {df_engineered.shape[1]:,}")
display(df_engineered.info())
display(df_engineered.isna().sum())
display(df_engineered.describe(include="all"))

Rows: 1,337
Columns: 7
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1337 entries, 0 to 1336
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1337 non-null   int64  
 1   sex       1337 non-null   object 
 2   bmi       1337 non-null   float64
 3   children  1337 non-null   int64  
 4   smoker    1337 non-null   object 
 5   region    1337 non-null   object 
 6   charges   1337 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.2+ KB


None

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

,age,sex,bmi,children,smoker,region,charges
count,1337.000000,1337,1337.000000,1337.000000,1337,1337,1337.000000
unique,NaN,2,NaN,NaN,2,4,NaN
top,NaN,male,NaN,NaN,no,southeast,NaN
freq,NaN,675,NaN,NaN,1063,364,NaN
mean,39.222139,NaN,30.663452,1.095737,NaN,NaN,13279.121487
std,14.044333,NaN,6.100468,1.205571,NaN,NaN,12110.359656
min,18.000000,NaN,15.960000,0.000000,NaN,NaN,1121.873900
25%,27.000000,NaN,26.290000,0.000000,NaN,NaN,4746.344000
50%,39.000000,NaN,30.400000,1.000000,NaN,NaN,9386.161300
75%,51.000000,NaN,34.700000,2.000000,NaN,NaN,16657.717450


## Derived Features

These engineered variables give models clearer signals.

- `bmi_category`: clinical-style BMI bands
- `is_obese`: flag for BMI at or above 30
- `age_group`: broad age bands
- `has_children`: whether dependents are listed
- `age_bmi_interaction`: combined age and BMI risk
- `smoker_bmi_interaction`: higher-risk combination of smoker status and BMI

In [23]:
df_engineered["is_obese"] = (df_engineered["bmi"] >= 30).astype(int)
df_engineered["has_children"] = (df_engineered["children"] > 0).astype(int)

In [24]:
df_engineered["bmi_category"] = pd.cut(
    df_engineered["bmi"],
    bins=[0, 18.5, 24.9, 29.9, 100],
    labels=["underweight", "normal", "overweight", "obese"],
)
df_engineered["age_group"] = pd.cut(
    df_engineered["age"],
    bins=[0, 25, 45, 65, 120],
    labels=["youth", "young_adult", "middle_aged", "senior"],
)

# Interactions (mapping smoker to 1/0 for the math interaction)
smoker_numeric = df_engineered["smoker"].map({"yes": 1, "no": 0})
df_engineered["age_bmi_interaction"] = (
    df_engineered["age"] * df_engineered["bmi"]
)
df_engineered["smoker_bmi_interaction"] = smoker_numeric * df_engineered["bmi"]

## Target transformation

`charges` is right-skewed (check EDA file) in this dataset. Keeping the original target is useful for interpretation, while `charges_log` is useful for regression models that perform better with a less skewed target.

In [25]:

df['charges_log'] = np.log(df['charges'])

print(f"Original charges skew: {df['charges'].skew():.3f}")
print(f"Log charges skew: {df['charges_log'].skew():.3f}")

Original charges skew: 1.515
Log charges skew: -0.090


## Encode features

Categorical variables are one-hot encoded with the first level dropped to reduce redundant columns.

In [26]:
numeric_features = [
    "age",
    "bmi",
    "children",
    "is_obese",
    "has_children",
    "age_bmi_interaction",
    "smoker_bmi_interaction",
]
categorical_features = ["sex", "smoker", "region", "bmi_category", "age_group"]

# drop_first=False matches the default OneHotEncoder behavior
df_encoded = pd.get_dummies(
    df_engineered,
    columns=["sex", "smoker", "region", "bmi_category", "age_group"],
    drop_first=True,
)


## Save Feature-Engineered Data

In [27]:
df_encoded.to_csv('../data/feature_engineered.csv')